# Milestone 16I — Live campaign monitor

**Read-only dashboard.** This notebook never writes to a campaign directory: it only reads
`live/status.json`, `history.csv`, `events.jsonl`, `event_summary.csv`, and
`live/latest_morphology.png`, all produced by the independent background simulation
process (`scripts/m16i_three_barrier_campaign.py`). Closing this notebook, restarting the
kernel, or an exception in a cell below has **no effect** on the running simulation,
its checkpoints, or its sink/hazard state — the simulation process is the sole writer.

Set `RUN_DIR` to the regime directory you want to watch (typically `.../finite` first,
per the addendum's finite-first execution order), then run the auto-refresh cell.

In [ ]:
import sys
sys.path.insert(0, "..")

RUN_DIR = "../runs/m16i_three_barrier_prod/finite"  # edit to point at the campaign/regime you want to watch
REFRESH_S = 60  # Section A13's default

In [ ]:
from IPython.display import display, clear_output
import matplotlib.pyplot as plt

from pf_sintering.live_dashboard import read_status, recent_events_table, render_dashboard_figure


def show_dashboard_once(run_dir=RUN_DIR):
    """Renders one snapshot of the dashboard without entering the
    auto-refresh loop -- useful for a single manual check."""
    status = read_status(run_dir)
    if status:
        print(f"{status.get('barrier_regime','?').upper()}  "
              f"t={status.get('simulation_time', float('nan')):.3f}  "
              f"step={status.get('step','?')}  events={status.get('event_count','?')}  "
              f"sink={'ACTIVE' if status.get('sink_active') else 'inactive'}  "
              f"hazard={status.get('hazard', float('nan')):.4e}  "
              f"sigma_s={status.get('sigma_s_1p5W_MPa', float('nan')):.4f} MPa  "
              f"X_neck={status.get('X_neck_nm', float('nan')):.3f} nm  "
              f"strain={status.get('sintering_strain', float('nan')):.4e}  "
              f"mass_drift={status.get('mass_drift', float('nan')):.2e}")
    else:
        print(f"no live/status.json yet at {run_dir} -- has the run started?")

    fig = render_dashboard_figure(run_dir)
    display(fig)
    plt.close(fig)

    events = recent_events_table(run_dir, n_recent=5)
    if events:
        print("\nMost recent events:")
        header = list(events[0].keys())
        print("  ".join(f"{h:>16s}" for h in header))
        for row in events:
            print("  ".join(f"{row[h]:>16s}" for h in header))


show_dashboard_once()

## Auto-refresh loop

Run the cell below to start auto-refreshing every `REFRESH_S` seconds. Interrupt the
kernel (Kernel → Interrupt, or the stop button) at any time to stop the loop — this only
stops the *notebook's* refreshing; the simulation keeps running untouched in the
background (Section A5/A26).

In [ ]:
import asyncio


async def live_monitor_loop(run_dir=RUN_DIR, refresh_s=REFRESH_S):
    try:
        while True:
            clear_output(wait=True)
            show_dashboard_once(run_dir)
            await asyncio.sleep(refresh_s)
    except asyncio.CancelledError:
        print("monitor loop stopped (simulation is unaffected)")


# Top-level await: run this cell, then interrupt the kernel to stop refreshing.
await live_monitor_loop()